<a href="https://colab.research.google.com/github/jaimeisaac2020/Python-analsisis-basicos/blob/mi-github/como_se_descargaron_los_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creación del Dataset para la Tesis

**Objetivo:** Este notebook recrea el dataset `dataset_completo_google_amazon.csv` obteniendo los datos de fuentes públicas y realizando los cálculos necesarios. Esto asegura la reproducibilidad de la investigación.

**Fuentes de Datos:**
- **Yahoo Finance:** Para precios de acciones (GOOGL, AMZN) e índices (S&P 500, NASDAQ, VIX).
- **FRED (Federal Reserve Economic Data):** Para variables macroeconómicas (Tasa del Tesoro a 10 años, Expectativas de Inflación).

**Variables Calculadas:**
- Retorno diario (`Return_1d`).
- Volatilidad de 20 días (`Volatility_20`).

In [1]:
!pip install pandas yfinance pandas_datareader

In [2]:
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import numpy as np

print("--- Proceso de Creación del Dataset Iniciado ---")

# --- 1. Definición de Parámetros ---
# Define el rango de fechas. Le damos un poco de margen al inicio para asegurar que el cálculo de la volatilidad sea correcto desde el primer día.
start_date = '2021-01-01'
end_date = '2024-12-31' # Puedes ajustar esta fecha al día actual si lo deseas

# Tickers para Yahoo Finance
stock_tickers = ['GOOGL', 'AMZN']
index_tickers = ['^VIX', '^GSPC', '^IXIC'] # VIX, S&P 500, NASDAQ

# IDs de series para la base de datos de la Reserva Federal (FRED)
fred_series = {
    'DGS10': 'Treasury_10Y',        # 10-Year Treasury Constant Maturity Rate
    'T5YIE': 'Inflacion_T5YIE'      # 5-Year, 5-Year Forward Inflation Expectation Rate
}

# --- 2. Descarga de Datos Financieros desde Yahoo Finance ---
print(f"\nDescargando datos de acciones e índices desde Yahoo Finance para el período {start_date} a {end_date}...")
try:
    # Descargamos todos los tickers en una sola llamada para mayor eficiencia
    data_yf = yf.download(stock_tickers + index_tickers, start=start_date, end=end_date)
    print("Datos de Yahoo Finance descargados exitosamente.")

    # Seleccionamos solo los precios de cierre y renombramos las columnas para mayor claridad
    prices = data_yf['Close'].copy()
    prices.rename(columns={
        'GOOGL': 'GOOGL_Close',
        'AMZN': 'AMZN_Close',
        '^GSPC': 'SP500',
        '^IXIC': 'NASDAQ',
        '^VIX': 'VIX'
    }, inplace=True)

except Exception as e:
    print(f"Error al descargar datos de Yahoo Finance: {e}")
    exit()

# --- 3. Descarga de Datos Macroeconómicos desde FRED ---
print(f"Descargando datos macroeconómicos desde FRED...")
try:
    # Descargamos los datos de FRED
    data_fred = web.DataReader(list(fred_series.keys()), 'fred', start=start_date, end=end_date)
    data_fred.rename(columns=fred_series, inplace=True)
    print("Datos de FRED descargados exitosamente.")
except Exception as e:
    print(f"Error al descargar datos de FRED: {e}")
    exit()

# --- 4. Combinación y Limpieza de Datos ---
print("\nCombinando y limpiando los datasets...")

# Unimos los dos DataFrames por el índice de fecha
df_combined = prices.join(data_fred, how='left')

# Los datos de FRED tienen valores solo para días laborables.
# Usamos forward fill para rellenar los fines de semana y festivos con el último valor conocido.
df_combined[list(fred_series.values())] = df_combined[list(fred_series.values())].ffill()

# Eliminamos cualquier fila que aún pueda tener NaNs (por ejemplo, el primer día si FRED no tenía datos)
df_combined.dropna(inplace=True)

print("Datos combinados y limpios.")

# --- 5. Cálculo de Variables Derivadas ---
print("Calculando retornos y volatilidad...")

# 5.1. Retornos Diarios
# Calcula el cambio porcentual de un día para otro
df_combined['GOOGL_Return_1d'] = df_combined['GOOGL_Close'].pct_change(1)
df_combined['AMZN_Return_1d'] = df_combined['AMZN_Close'].pct_change(1)

# 5.2. Volatilidad Móvil a 20 días
# Calcula la desviación estándar de los últimos 20 precios de cierre
window = 20
df_combined['GOOGL_Volatility_20'] = df_combined['GOOGL_Close'].rolling(window=window).std()
df_combined['AMZN_Volatility_20'] = df_combined['AMZN_Close'].rolling(window=window).std()

# Eliminamos las filas iniciales que tienen NaN debido al cálculo de la ventana móvil
df_final = df_combined.dropna()
# Reajustamos el índice para que empiece de nuevo si es necesario
# df_final.reset_index(inplace=True)

print("Variables derivadas calculadas.")

# --- 6. Formateo y Guardado del Archivo Final ---
print("\nFormateando y guardando el archivo final...")

# Reordenamos las columnas para que coincidan con el formato del archivo original
final_columns_order = [
    'GOOGL_Close', 'AMZN_Close', 'Treasury_10Y', 'VIX', 'SP500', 'NASDAQ',
    'GOOGL_Return_1d', 'AMZN_Return_1d', 'GOOGL_Volatility_20',
    'AMZN_Volatility_20', 'Inflacion_T5YIE'
]
df_final = df_final[final_columns_order]

# Guardamos el DataFrame en un nuevo archivo CSV
output_filename = 'dataset_recreado.csv'
df_final.to_csv(output_filename)

print(f"\n¡Proceso completado! El dataset ha sido guardado como '{output_filename}'")
print("\nVista previa de las últimas 5 filas del dataset generado:")
print(df_final.tail())

--- Proceso de Creación del Dataset Iniciado ---

Descargando datos de acciones e índices desde Yahoo Finance para el período 2021-01-01 a 2024-12-31...


/tmp/ipython-input-249388944.py:27: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data_yf = yf.download(stock_tickers + index_tickers, start=start_date, end=end_date)
[*********************100%***********************]  5 of 5 completed


Datos de Yahoo Finance descargados exitosamente.
Descargando datos macroeconómicos desde FRED...
Datos de FRED descargados exitosamente.

Combinando y limpiando los datasets...
Datos combinados y limpios.
Calculando retornos y volatilidad...
Variables derivadas calculadas.

Formateando y guardando el archivo final...

¡Proceso completado! El dataset ha sido guardado como 'dataset_recreado.csv'

Vista previa de las últimas 5 filas del dataset generado:
            GOOGL_Close  AMZN_Close  Treasury_10Y        VIX        SP500  \
Date                                                                        
2024-12-23   193.997543  225.059998          4.59  16.780001  5974.069824   
2024-12-24   195.472717  229.050003          4.59  14.270000  6040.040039   
2024-12-26   194.964386  227.050003          4.58  14.730000  6037.589844   
2024-12-27   192.133591  223.750000          4.62  15.950000  5970.839844   
2024-12-30   190.618561  221.300003          4.55  17.400000  5906.939941   

    

In [3]:
# 7. Formateo y Guardado del Archivo Final
print("Formateando y guardando el archivo final...")

# Reordenamos las columnas para que coincidan con el formato del archivo original
final_columns_order = [
    'GOOGL_Close', 'AMZN_Close', 'Treasury_10Y', 'VIX', 'SP500', 'NASDAQ',
    'GOOGL_Return_1d', 'AMZN_Return_1d', 'GOOGL_Volatility_20',
    'AMZN_Volatility_20', 'Inflacion_T5YIE'
]
df_final = df_final[final_columns_order]

# Guardamos el DataFrame en un nuevo archivo CSV
output_filename = 'dataset_recreado.csv'
df_final.to_csv(output_filename)

print(f"\n¡Proceso completado! El dataset ha sido guardado como '{output_filename}'")
print("\nVista previa de las últimas 5 filas del dataset generado:")
df_final.tail()

Formateando y guardando el archivo final...

¡Proceso completado! El dataset ha sido guardado como 'dataset_recreado.csv'

Vista previa de las últimas 5 filas del dataset generado:


,GOOGL_Close,AMZN_Close,Treasury_10Y,VIX,SP500,NASDAQ,GOOGL_Return_1d,AMZN_Return_1d,GOOGL_Volatility_20,AMZN_Volatility_20,Inflacion_T5YIE
Date,,,,,,,,,,,
2024-12-23,193.997543,225.059998,4.59,16.780001,5974.069824,19764.880859,0.016823,0.000622,10.925744,9.393671,2.38
2024-12-24,195.472717,229.050003,4.59,14.270000,6040.040039,20031.130859,0.007604,0.017729,10.862944,8.437937,2.39
2024-12-26,194.964386,227.050003,4.58,14.730000,6037.589844,20020.359375,-0.002601,-0.008732,10.686817,7.838409,2.39
2024-12-27,192.133591,223.750000,4.62,15.950000,5970.839844,19722.029297,-0.014520,-0.014534,10.213698,6.736610,2.38
2024-12-30,190.618561,221.300003,4.55,17.400000,5906.939941,19486.789062,-0.007885,-0.010950,9.492955,5.666061,2.35
